### Pinecone Vector DB



In [ ]:
api_key = "####"

In [5]:
!uv add  langchain langchain-pinecone langchain-openai

Resolved 195 packages in 173ms
   Building rag @ file:///D:/Mission%20Project/RAG
      Built rag @ file:///D:/Mission%20Project/RAG
Prepared 8 packages in 2.35s
Uninstalled 2 packages in 10ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 8 packages in 418ms
 + aiohttp-retry==2.9.1
 + langchain-pinecone==0.2.13
 - packaging==26.3
 + packaging==24.2
 + pinecone==7.3.0
 + pinecone-plugin-assistant==1.8.0
 + pinecone-plugin-interface==0.0.7
 ~ rag==0.1.0 (from file:///D:/Mission%20Project/RAG)
 + simsimd==6.5.16


In [6]:
from pinecone import Pinecone
pc=Pinecone(api_key=api_key)

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings= HuggingFaceEmbeddings(
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2389.92it/s]


In [20]:
from pinecone import ServerlessSpec

index_name = "rag"

if not pc.has_index(index_name):
    pc.create_index(
        name= index_name,
        metric="cosine",
        dimension=384,
        spec= ServerlessSpec(cloud="aws", region="us-east-1")
    )
index = pc.Index(index_name)

In [21]:
index

In [22]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [23]:
vector_store

In [24]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [26]:
index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 0}, 'rag': {'vector_count': 0}},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [28]:
pc.describe_index("rag")

{
    "name": "rag",
    "metric": "cosine",
    "host": "rag-ap5ds36.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null
}

In [29]:
from pinecone import ServerlessSpec

index_name = "rag-384"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [33]:
from langchain_pinecone import PineconeVectorStore

index_name = "rag-384"

index = pc.Index(index_name)

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [34]:
print(vector_store.index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [35]:
vector_store.add_documents(documents=documents)

['baa6d486-8d99-4b47-a90a-fe6d00f85bed',
 'aba6a17a-ae72-4af7-b6cc-d54b02ff1926',
 '2fbd7c70-5616-4625-8fae-804af6e3886b',
 'e395c365-194f-42d1-bd88-fa6f578d85a2',
 'dd178b4e-492b-4f87-8613-de547fb74f77',
 'cd4c03e2-de60-4173-bced-2f95168aa4cd',
 '5cde00f3-5827-4e67-acdd-3ee70f9903f0',
 '8f9c2b8d-ae58-4986-8a3b-9dabbd3ca657',
 'abafa1cb-7fec-4866-b11e-fab615f63621',
 '7d2cb68d-e970-453a-a6cd-5ac49f117874']

In [36]:
### Query Directly
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [37]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.541721] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [38]:
### Retriever
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='e395c365-194f-42d1-bd88-fa6f578d85a2', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]